In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import ast
import matplotlib.pyplot as plt
from tabulate import tabulate

In [14]:
dataset = ["lcstep", "recipenlg", "champ"]
embedder = ["hf-all-mpnet-base-v2", "openai-text-embedding-3-large"]
system = ["rag", "aag"]

### Eval Heuristics

In [ ]:
for sys in system:
    for data in dataset:
        for embed in embedder:
            eval_results_file = f"../output/{sys}/{data}/{embed}/eval_results.csv"
            eval_metrics = pd.read_csv(eval_results_file, header=0)
            eval_metrics = eval_metrics.drop(eval_metrics[eval_metrics.Overall < 0].index)
            if 'ROUGE' in list(eval_metrics):
                eval_metrics = eval_metrics.drop(columns=['ROUGE'])
            print(f"Printing mean stats for output/{sys}/{data}/{embed}: {eval_metrics.mean()}")

In [10]:
for sys in system:
    for data in dataset:
        for embed in embedder:
            eval_results_file = f"../output/{sys}/{data}/{embed}/eval_results.csv"
            eval_metrics = pd.read_csv(eval_results_file, header=0).drop(columns=['_id'])
            eval_metrics = eval_metrics.drop(eval_metrics[eval_metrics.Overall < 0].index)
            if 'ROUGE' in list(eval_metrics):
                eval_metrics['ROUGE'] = eval_metrics['ROUGE'].apply(lambda x: ast.literal_eval(x))
                rouge_df = eval_metrics['ROUGE'].apply(pd.Series)
                rouge_df = rouge_df[['rouge1_fmeasure', 'rouge2_fmeasure', 'rougeL_fmeasure', 'rougeLsum_fmeasure']]
                new_df = pd.concat([eval_metrics.drop(columns=["ROUGE"]), rouge_df], axis=1)
                print(f"Printing mean stats for output/{sys}/{data}/{embed}: {new_df.mean()}")
            else:
                print(f"Printing mean stats for output/{sys}/{data}/{embed}: {eval_metrics.mean()}")

Printing mean stats for output/rag/lcstep/hf-all-mpnet-base-v2: Api-Overlap           0.303395
TfIdf                 0.396296
Inp-Used              0.518519
Overall               4.222222
rouge1_fmeasure       0.016524
rouge2_fmeasure       0.000245
rougeL_fmeasure       0.014738
rougeLsum_fmeasure    0.016474
dtype: float64
Printing mean stats for output/rag/lcstep/openai-text-embedding-3-large: Api-Overlap           0.311199
TfIdf                 0.446296
Inp-Used              0.537037
Overall               4.296296
rouge1_fmeasure       0.020234
rouge2_fmeasure       0.000537
rougeL_fmeasure       0.017539
rougeLsum_fmeasure    0.020234
dtype: float64
Printing mean stats for output/rag/recipenlg/hf-all-mpnet-base-v2: TfIdf                 0.266000
Ing_Used              0.803304
Edit-Distance         9.260000
Num-Compare           0.057429
Overall               4.830000
rouge1_fmeasure       0.043918
rouge2_fmeasure       0.001305
rougeL_fmeasure       0.031102
rougeLsum_fmeasure    

In [ ]:
for ev_metric_name in list(eval_metrics.columns.values):
    if ev_metric_name == "_id" or ev_metric_name == "Overall":
        continue
    sp_corr = stats.spearmanr(eval_metrics[ev_metric_name], eval_metrics['Overall']/10)
    print(f"Spearman Coefficient of Overall Score and {ev_metric_name} is: {sp_corr.statistic}")

In [ ]:
plt.scatter(eval_metrics['Ing_Used'].to_numpy(), eval_metrics['Overall'].to_numpy())

### Pairwise Evals

In [13]:

##without GT steps

res = {"Datasets":dataset, "Total":[27, 100, 27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    for data in dataset:
        p_eval_file = f'../output/pair-eval-without-gt/rag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        df = df.drop(df[df.choice < 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(tot-len(choices))
    res[emb] = emb_res
    res[f"unparsed_{service}"] = emb_unparsed

print(tabulate(res, headers="keys", tablefmt="grid"))

+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| Datasets   |   Total |   hf-all-mpnet-base-v2 |   unparsed_hf |   openai-text-embedding-3-large |   unparsed_openai |
+============+=========+========================+===============+=================================+===================+
| lcstep     |      27 |                      1 |             0 |                               1 |                 0 |
+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| recipenlg  |     100 |                      4 |             2 |                              15 |                 3 |
+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| champ      |      27 |                     10 |             1 |                              -3 |                 0 |
+------------+---------+----------------

In [15]:

##without GT steps - 5 runs panel strategy
# dataset = ["recipenlg"]
res = {"Datasets":dataset, "Total":[27,100,27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    emb_tie = []
    for data in dataset:
        p_eval_file = f'../output/without-gt/rag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        choices_orig = df['choice'].to_numpy()
        df = df.drop(df[df.choice <= 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(np.sum(choices_orig < 0))
        emb_tie.append(np.sum(choices_orig == 0))
    res[emb] = emb_res
    res[f"unparsed_{service}"] = emb_unparsed
    res[f"ties_{service}"] = emb_tie

print(tabulate(res, headers="keys", tablefmt="grid"))

+------------+---------+------------------------+---------------+-----------+---------------------------------+-------------------+---------------+
| Datasets   |   Total |   hf-all-mpnet-base-v2 |   unparsed_hf |   ties_hf |   openai-text-embedding-3-large |   unparsed_openai |   ties_openai |
+============+=========+========================+===============+===========+=================================+===================+===============+
| lcstep     |      27 |                     -8 |             0 |         7 |                               4 |                 0 |             7 |
+------------+---------+------------------------+---------------+-----------+---------------------------------+-------------------+---------------+
| recipenlg  |     100 |                     -1 |             0 |        15 |                               7 |                 0 |             9 |
+------------+---------+------------------------+---------------+-----------+---------------------------------+-

In [ ]:

##without GT steps - 5 runs panel strategy - between embeds
# dataset = ["recipenlg"]
res = {"Datasets":dataset, "Total":[27,100,27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    emb_tie = []
    for data in dataset:
        p_eval_file = f'../output/without-gt-between-embeds/aag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        choices_orig = df['choice'].to_numpy()
        df = df.drop(df[df.choice <= 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(np.sum(choices_orig < 0))
        emb_tie.append(np.sum(choices_orig == 0))
    res["score"] = emb_res
    res[f"unparsed"] = emb_unparsed
    res[f"ties"] = emb_tie
    break

print(tabulate(res, headers="keys", tablefmt="grid"))

In [16]:

##with GT steps
res = {"Datasets":dataset, "Total":[27, 100, 27]}
for emb in embedder:
    service, model = emb.split("-", maxsplit=1)
    emb_res = []
    emb_unparsed = []
    for data in dataset:
        p_eval_file = f'../output/with-gt/rag_aag_{data}_{emb}_pair_eval.csv'
        df = pd.read_csv(p_eval_file, header=0, usecols=["question_id", "choice"])
        tot = len(df)
        df = df.drop(df[df.choice < 0].index)
        choices = df['choice'].to_numpy()
        num_aag = np.sum(choices-1)
        num_rag = len(choices) - num_aag
        emb_res.append(num_aag-num_rag)
        emb_unparsed.append(tot-len(choices))
    res[emb] = emb_res
    res[f"unparsed_{service}"] = emb_unparsed

print(tabulate(res, headers="keys", tablefmt="grid"))

+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| Datasets   |   Total |   hf-all-mpnet-base-v2 |   unparsed_hf |   openai-text-embedding-3-large |   unparsed_openai |
+============+=========+========================+===============+=================================+===================+
| lcstep     |      27 |                     -3 |             2 |                              -4 |                 1 |
+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| recipenlg  |     100 |                     -2 |             2 |                              20 |                 4 |
+------------+---------+------------------------+---------------+---------------------------------+-------------------+
| champ      |      27 |                      1 |             0 |                              -5 |                 0 |
+------------+---------+----------------